<a href="https://colab.research.google.com/github/PatrickHuynh837/MMAlytics/blob/main/model.ip_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup

In [1]:
# Install psycopg if not already installed
!pip install sqlalchemy pandas "psycopg[binary]"
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 28.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 213.0/213.0 kB 9.8 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import psycopg2

DB_URL = (
    "postgresql+psycopg://neondb_owner:npg_Bo2SUY6ngypR@"
    "ep-orange-frost-afcl94sd-pooler.c-2.us-west-2.aws.neon.tech/"
    "neondb?sslmode=require"
)

engine = create_engine(
        DB_URL,
        pool_pre_ping=True,
        pool_recycle=3600
        )

In [3]:
#Loading Data
df = pd.read_sql(
    """
    SELECT *
    FROM ml.fight_dataset
    """,
    engine
)

df.head()

,fight_url,event_url,event_name,event_date,location_city,location_state,location_country,referee,weight_class,gender,...,fighter_2_takedown_att,fighter_2_takedown_succ,fighter_2_submission_att,fighter_2_reversals,fighter_2_ctrl_time,fighter_1_rank,fighter_2_rank,fighter_1_odds,fighter_2_odds,created_at
0,http://ufcstats.com/fight-details/c13dc0cccef2...,http://ufcstats.com/event-details/31e1ea6fe6b6...,UFC Fight Night: Fiziev vs. Torres,2026-06-27,Baku,Azerbaijan,None,Marc Goddard,Lightweight,M,...,0.0,0.0,0.0,0.0,0:06,11.0,15.0,NaN,NaN,2026-06-29 05:32:18.095360
1,http://ufcstats.com/fight-details/81cde317c156...,http://ufcstats.com/event-details/31e1ea6fe6b6...,UFC Fight Night: Fiziev vs. Torres,2026-06-27,Baku,Azerbaijan,None,Herb Dean,Flyweight,M,...,0.0,0.0,1.0,0.0,0:00,8.0,14.0,NaN,NaN,2026-06-29 05:32:18.095360
2,http://ufcstats.com/fight-details/809814f03ff3...,http://ufcstats.com/event-details/31e1ea6fe6b6...,UFC Fight Night: Fiziev vs. Torres,2026-06-27,Baku,Azerbaijan,None,Rich Mitchell,Lightweight,M,...,0.0,0.0,0.0,0.0,0:03,NaN,NaN,NaN,NaN,2026-06-29 05:32:18.095360
3,http://ufcstats.com/fight-details/012c307c9d44...,http://ufcstats.com/event-details/31e1ea6fe6b6...,UFC Fight Night: Fiziev vs. Torres,2026-06-27,Baku,Azerbaijan,None,Herb Dean,Middleweight,M,...,2.0,0.0,0.0,0.0,4:04,NaN,14.0,NaN,NaN,2026-06-29 05:32:18.095360
4,http://ufcstats.com/fight-details/6a35ff33b859...,http://ufcstats.com/event-details/31e1ea6fe6b6...,UFC Fight Night: Fiziev vs. Torres,2026-06-27,Baku,Azerbaijan,None,Rich Mitchell,Welterweight,M,...,0.0,0.0,0.0,0.0,0:03,NaN,NaN,NaN,NaN,2026-06-29 05:32:18.095360


# Scope

In [ ]:

#Find missing data
missing = (
    df.isna()
      .sum()
      .sort_values(ascending=False)
      .to_frame('missing_count')
)

missing

,missing_count
fighter_2_rank,7030
fighter_1_rank,6163
fighter_1_odds,2486
fighter_2_odds,2486
fighter_2_reach_cm,908
...,...
fighter_2_sapm,0
fighter_2_td_def,0
fighter_2_td_acc,0
fighter_2_sub_avg,0


In [ ]:
list(df.columns)

['fight_url',
 'event_url',
 'event_name',
 'event_date',
 'location_city',
 'location_state',
 'location_country',
 'referee',
 'weight_class',
 'gender',
 'title_fight',
 'num_rounds',
 'fighter_1',
 'fighter_2',
 'fighter_1_url',
 'fighter_2_url',
 'winner',
 'result',
 'result_details',
 'finish_round',
 'finish_time',
 'fighter_1_height_cm',
 'fighter_1_weight_lbs',
 'fighter_1_reach_cm',
 'fighter_1_stance',
 'fighter_1_dob',
 'fighter_2_height_cm',
 'fighter_2_weight_lbs',
 'fighter_2_reach_cm',
 'fighter_2_stance',
 'fighter_2_dob',
 'fighter_1_wins',
 'fighter_1_losses',
 'fighter_1_draws',
 'fighter_1_slpm',
 'fighter_1_str_acc',
 'fighter_1_sapm',
 'fighter_1_str_def',
 'fighter_1_td_avg',
 'fighter_1_td_acc',
 'fighter_1_td_def',
 'fighter_1_sub_avg',
 'fighter_2_wins',
 'fighter_2_losses',
 'fighter_2_draws',
 'fighter_2_slpm',
 'fighter_2_str_acc',
 'fighter_2_sapm',
 'fighter_2_str_def',
 'fighter_2_td_avg',
 'fighter_2_td_acc',
 'fighter_2_td_def',
 'fighter_2_sub_avg',

# Data Cleaning




## Exploration

### Reach

In [ ]:
missing_fighters = df.loc[
    df["fighter_1_reach_cm"].isna(),
    "fighter_1"
].unique()

fighters_in_both = [
    fighter for fighter in missing_fighters
    if ((df["fighter_1"] == fighter) | (df["fighter_2"] == fighter)).sum() > 1
]

fighters_in_both

['Din Thomas',
 'David Michaud',
 'Ben Wall',
 'Gan McGee',
 'Jutaro Nakao',
 'Geza Kalman',
 'Marcio Cruz',
 'Mark Hall',
 'Royce Gracie',
 'Remco Pardoel',
 'Patrick Smith',
 'Nick Penner',
 'Sam Adkins',
 'Horacio Gutierrez',
 'Matt Van Buren',
 'Tateki Matsuda',
 'Charlie Ward',
 'Luiz Dutra',
 'Nolan Ticman',
 'Wagner Silva',
 'Papy Abedi',
 'Kelly Faszholz',
 'Nate Loughran',
 'Steve Montgomery',
 'War Machine',
 'Tim Lajcik',
 'Don Frye',
 'Cal Worsham',
 'Keith Hackney',
 'Kwan Ho Kwak',
 'Alan Omer',
 'Wendell Oliveira Marques',
 'Ricardo Abreu',
 'Tor Troeng',
 'Marcus Bossett',
 'Juan Manuel Puig',
 'Ron Waterman',
 'Mike Lullo',
 'Paul Jones',
 'Mark Kerr',
 'Paul Varelans',
 'Carlos Newton',
 'Genki Sudo',
 'Joe Charles',
 'Jason DeLucia',
 'Mike van Arsdale',
 'Kevin Jackson',
 'Ivan Salaverry',
 'Semmy Schilt',
 'Anying Wang',
 'Laverne Clark',
 'Maurice Smith',
 'Brian Johnston',
 'Gary Goodridge',
 'Kevin Randleman',
 'David Abbott',
 'Kimo Leopoldo',
 'Anthony Macias'

In [ ]:
missing_urls = pd.concat([
    df.loc[df["fighter_1_reach_cm"].isna(), "fighter_1_url"],
    df.loc[df["fighter_2_reach_cm"].isna(), "fighter_2_url"]
]).dropna().unique()

print(len(missing_urls))

666


### Weight

In [ ]:
df[df["fighter_1_weight_lbs"].isna()][
    ["fighter_1","fighter_1_url","weight_class"]
]

,fighter_1,fighter_1_url,weight_class
5696,Cesar Marscucci,http://ufcstats.com/fighter-details/e8efeb9cf3...,Lightweight
5934,Frank Hamaker,http://ufcstats.com/fighter-details/c3c23c9947...,Open Weight
6059,Jack Nilson,http://ufcstats.com/fighter-details/53e533db1b...,Lightweight


In [ ]:
df[df["fighter_2_weight_lbs"].isna()][
    ["fighter_2","fighter_2_url","weight_class"]
]

,fighter_2,fighter_2_url,weight_class
1300,Frank Caracci,http://ufcstats.com/fighter-details/44f9c777fe...,Lightweight
1320,Sam Fulton,http://ufcstats.com/fighter-details/1f5f756585...,Heavyweight
1334,Jack Nilson,http://ufcstats.com/fighter-details/53e533db1b...,Open Weight
1373,He-Man Gipson,http://ufcstats.com/fighter-details/3e03cec976...,Open Weight
2108,Felix Lee Mitchell,http://ufcstats.com/fighter-details/6cbb7661c3...,Open Weight
3378,Eric Martin,http://ufcstats.com/fighter-details/abbc4fc02e...,Heavyweight
3387,Dave Berry,http://ufcstats.com/fighter-details/b3bbe88fec...,Open Weight
3432,Jack McGlaughlin,http://ufcstats.com/fighter-details/237187ed9f...,Open Weight
5681,Sam Fulton,http://ufcstats.com/fighter-details/1f5f756585...,Open Weight
5696,Paulo Santos,http://ufcstats.com/fighter-details/7ca4c3f8aa...,Lightweight


## Preprocessing

### Rankings

In [4]:

#Classify Ranked or Unranked
df["fighter_1_is_ranked"] = (
    df["fighter_1_rank"].between(0, 15)
).astype(int)

df["fighter_2_is_ranked"] = (
    df["fighter_2_rank"].between(0, 15)
).astype(int)

#Fill in ranks
df["fighter_1_rank"] = df["fighter_1_rank"].fillna(16)
df["fighter_2_rank"] = df["fighter_2_rank"].fillna(16)

### Weight

In [5]:
weight_map = {
    "Flyweight": 125,
    "Bantamweight": 135,
    "Featherweight": 145,
    "Lightweight": 155,
    "Welterweight": 170,
    "Middleweight": 185,
    "Light Heavyweight": 205,

}

df["fighter_1_weight_lbs"] = df["fighter_1_weight_lbs"].fillna(
    df["weight_class"].map(weight_map)
)

df["fighter_2_weight_lbs"] = df["fighter_2_weight_lbs"].fillna(
    df["weight_class"].map(weight_map)
)

### Label

In [6]:
df["fighter_1_win"] = (df["winner"] == df["fighter_1"]).astype(int)

### Fight *Stats*

In [7]:
# Create unique fight identifier
df = df.reset_index(names="fight_id")



# -------------------------
# Build long fighter history
# -------------------------
fighter_1_history = (
    df[
        [
            "fight_id",
            "event_date",

            # identity
            "fighter_1",
            "fighter_2",
            "winner",
            "result",

            # physical
            "fighter_1_height_cm",
            "fighter_1_weight_lbs",
            "fighter_1_reach_cm",
            "fighter_1_stance",
            "fighter_1_dob",

            # career stats
            "fighter_1_wins",
            "fighter_1_losses",
            "fighter_1_draws",
            "fighter_1_rank",
            "fighter_1_odds",
            "fighter_1_is_ranked",

            # striking/grappling style
            "fighter_1_slpm",
            "fighter_1_str_acc",
            "fighter_1_sapm",
            "fighter_1_str_def",
            "fighter_1_td_avg",
            "fighter_1_td_acc",
            "fighter_1_td_def",
            "fighter_1_sub_avg",

            # fight output stats
            "fighter_1_knockdowns",
            "fighter_1_sig_strikes_att",
            "fighter_1_sig_strikes_succ",
            "fighter_1_total_strikes_att",
            "fighter_1_total_strikes_succ",
            "fighter_1_takedown_att",
            "fighter_1_takedown_succ",
            "fighter_1_submission_att",
            "fighter_1_reversals",
            "fighter_1_ctrl_time",
        ]
    ]
    .rename(columns={
        "fighter_1": "fighter",
        "fighter_2": "opponent",

        "fighter_1_height_cm": "height_cm",
        "fighter_1_weight_lbs": "weight_lbs",
        "fighter_1_reach_cm": "reach_cm",
        "fighter_1_stance": "stance",
        "fighter_1_dob": "dob",

        "fighter_1_wins": "career_wins",
        "fighter_1_losses": "career_losses",
        "fighter_1_draws": "career_draws",
        "fighter_1_rank": "rank",
        "fighter_1_odds": "odds",
        "fighter_1_is_ranked": "is_ranked",

        "fighter_1_slpm": "slpm",
        "fighter_1_str_acc": "str_acc",
        "fighter_1_sapm": "sapm",
        "fighter_1_str_def": "str_def",
        "fighter_1_td_avg": "td_avg",
        "fighter_1_td_acc": "td_acc",
        "fighter_1_td_def": "td_def",
        "fighter_1_sub_avg": "sub_avg",

        "fighter_1_knockdowns": "knockdowns",
        "fighter_1_sig_strikes_att": "sig_str_att",
        "fighter_1_sig_strikes_succ": "sig_str_succ",
        "fighter_1_total_strikes_att": "total_str_att",
        "fighter_1_total_strikes_succ": "total_str_succ",
        "fighter_1_takedown_att": "td_att",
        "fighter_1_takedown_succ": "td_succ",
        "fighter_1_submission_att": "sub_att",
        "fighter_1_reversals": "reversals",
        "fighter_1_ctrl_time": "ctrl_time",
    })
)

fighter_2_history = (
    df[
        [
            "fight_id",
            "event_date",

            # identity
            "fighter_1",
            "fighter_2",
            "winner",
            "result",

            # physical
            "fighter_2_height_cm",
            "fighter_2_weight_lbs",
            "fighter_2_reach_cm",
            "fighter_2_stance",
            "fighter_2_dob",

            # career stats
            "fighter_2_wins",
            "fighter_2_losses",
            "fighter_2_draws",
            "fighter_2_rank",
            "fighter_2_odds",
            "fighter_2_is_ranked",

            # striking/grappling style
            "fighter_2_slpm",
            "fighter_2_str_acc",
            "fighter_2_sapm",
            "fighter_2_str_def",
            "fighter_2_td_avg",
            "fighter_2_td_acc",
            "fighter_2_td_def",
            "fighter_2_sub_avg",

            # fight output stats
            "fighter_2_knockdowns",
            "fighter_2_sig_strikes_att",
            "fighter_2_sig_strikes_succ",
            "fighter_2_total_strikes_att",
            "fighter_2_total_strikes_succ",
            "fighter_2_takedown_att",
            "fighter_2_takedown_succ",
            "fighter_2_submission_att",
            "fighter_2_reversals",
            "fighter_2_ctrl_time",
        ]
    ]
    .rename(columns={
        "fighter_2": "fighter",
        "fighter_1": "opponent",

        "fighter_2_height_cm": "height_cm",
        "fighter_2_weight_lbs": "weight_lbs",
        "fighter_2_reach_cm": "reach_cm",
        "fighter_2_stance": "stance",
        "fighter_2_dob": "dob",

        "fighter_2_wins": "career_wins",
        "fighter_2_losses": "career_losses",
        "fighter_2_draws": "career_draws",
        "fighter_2_rank": "rank",
        "fighter_2_odds": "odds",
        "fighter_2_is_ranked": "is_ranked",

        "fighter_2_slpm": "slpm",
        "fighter_2_str_acc": "str_acc",
        "fighter_2_sapm": "sapm",
        "fighter_2_str_def": "str_def",
        "fighter_2_td_avg": "td_avg",
        "fighter_2_td_acc": "td_acc",
        "fighter_2_td_def": "td_def",
        "fighter_2_sub_avg": "sub_avg",

        "fighter_2_knockdowns": "knockdowns",
        "fighter_2_sig_strikes_att": "sig_str_att",
        "fighter_2_sig_strikes_succ": "sig_str_succ",
        "fighter_2_total_strikes_att": "total_str_att",
        "fighter_2_total_strikes_succ": "total_str_succ",
        "fighter_2_takedown_att": "td_att",
        "fighter_2_takedown_succ": "td_succ",
        "fighter_2_submission_att": "sub_att",
        "fighter_2_reversals": "reversals",
        "fighter_2_ctrl_time": "ctrl_time",
    })
)

history = pd.concat(
    [fighter_1_history, fighter_2_history],
    ignore_index=True
)

def time_to_seconds(x):
    if pd.isna(x):
        return 0
    parts = x.split(":")

    if len(parts) == 2:  # MM:SS
        m, s = parts
        return int(m) * 60 + int(s)
    elif len(parts) == 3:  # H:MM:SS (just in case)
        h, m, s = parts
        return int(h) * 3600 + int(m) * 60 + int(s)
    else:
        return 0

history["ctrl_time"] = history["ctrl_time"].apply(time_to_seconds)

history = history.sort_values(["fighter", "event_date", "fight_id"])

#### Record

In [8]:
# -------------------------
# Win/Loss indicators
# -------------------------

history["win"] = (
    history["fighter"] == history["winner"]
).astype(int)

history["loss"] = 1 - history["win"]

history["draw"] = (
    history["winner"] == "Draw"
).astype(int) if "Draw" in history["winner"].values else 0

# -------------------------
# Prior cumulative stats
# -------------------------

history["prior_wins"] = (
    history.groupby("fighter")["win"]
    .transform(lambda s: s.cumsum().shift(fill_value=0))
)

history["prior_losses"] = (
    history.groupby("fighter")["loss"]
    .transform(lambda s: s.cumsum().shift(fill_value=0))
)

history["prior_draws"] = (
    history.groupby("fighter")["draw"]
    .transform(lambda s: s.cumsum().shift(fill_value=0))
)

fighter_1_features = (
    history[
        ["fight_id", "fighter", "prior_wins", "prior_losses", "prior_draws"]
    ]
    .rename(columns={
        "fighter": "fighter_1",
        "prior_wins": "fighter_1_prior_wins",
        "prior_losses": "fighter_1_prior_losses",
        "prior_draws": "fighter_1_prior_draws"
    })
)

fighter_2_features = (
    history[
        ["fight_id", "fighter", "prior_wins", "prior_losses", "prior_draws"]
    ]
    .rename(columns={
        "fighter": "fighter_2",
        "prior_wins": "fighter_2_prior_wins",
        "prior_losses": "fighter_2_prior_losses",
        "prior_draws": "fighter_2_prior_draws"
    })
)

df = df.merge(
    fighter_1_features,
    on=["fight_id", "fighter_1"],
    how="left"
)

df = df.merge(
    fighter_2_features,
    on=["fight_id", "fighter_2"],
    how="left"
)

#### Striking

In [9]:
history = history.sort_values(["fighter", "event_date"])

history["slpm_roll_3"] = (
    history.groupby("fighter")["slpm"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

history["sapm_roll_3"] = (
    history.groupby("fighter")["sapm"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

history["str_acc_roll_3"] = (
    history.groupby("fighter")["str_acc"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

history["str_def_roll_3"] = (
    history.groupby("fighter")["str_def"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

fighter_1_features = (
    history[
        [
            "fight_id",
            "fighter",
            "slpm_roll_3",
            "sapm_roll_3",
            "str_acc_roll_3",
            "str_def_roll_3",
        ]
    ]
    .rename(columns={
        "fighter": "fighter_1",
        "slpm_roll_3": "fighter_1_slpm_roll_3",
        "sapm_roll_3": "fighter_1_sapm_roll_3",
        "str_acc_roll_3": "fighter_1_str_acc_roll_3",
        "str_def_roll_3": "fighter_1_str_def_roll_3",
    })
)

fighter_2_features = (
    history[
        [
            "fight_id",
            "fighter",
            "slpm_roll_3",
            "sapm_roll_3",
            "str_acc_roll_3",
            "str_def_roll_3",
        ]
    ]
    .rename(columns={
        "fighter": "fighter_2",
        "slpm_roll_3": "fighter_2_slpm_roll_3",
        "sapm_roll_3": "fighter_2_sapm_roll_3",
        "str_acc_roll_3": "fighter_2_str_acc_roll_3",
        "str_def_roll_3": "fighter_2_str_def_roll_3",
    })
)

df = df.merge(
    fighter_1_features,
    on=["fight_id", "fighter_1"],
    how="left"
)

df = df.merge(
    fighter_2_features,
    on=["fight_id", "fighter_2"],
    how="left"
)

#### Grappling

In [10]:
# -------------------------
# Grappling rolling features
# -------------------------

history["td_avg_roll_3"] = (
    history.groupby("fighter")["td_avg"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

history["td_acc_roll_3"] = (
    history.groupby("fighter")["td_acc"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

history["td_def_roll_3"] = (
    history.groupby("fighter")["td_def"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

history["sub_avg_roll_3"] = (
    history.groupby("fighter")["sub_avg"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

history["ctrl_time_roll_3"] = (
    history.groupby("fighter")["ctrl_time"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
)

history["td_success_rate_roll_3"] = (
    history.groupby("fighter")["td_succ"]
    .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
    / (
        history.groupby("fighter")["td_att"]
        .transform(lambda s: s.shift(1).rolling(3, min_periods=1).mean())
        + 1e-6
    )
)

fighter_1_features = (
    history[
        [
            "fight_id",
            "fighter",

            "td_avg_roll_3",
            "td_acc_roll_3",
            "td_def_roll_3",
            "sub_avg_roll_3",
            "ctrl_time_roll_3",
            "td_success_rate_roll_3",
        ]
    ]
    .rename(columns={
        "fighter": "fighter_1",
        "td_avg_roll_3": "fighter_1_td_avg_roll_3",
        "td_acc_roll_3": "fighter_1_td_acc_roll_3",
        "td_def_roll_3": "fighter_1_td_def_roll_3",
        "sub_avg_roll_3": "fighter_1_sub_avg_roll_3",
        "ctrl_time_roll_3": "fighter_1_ctrl_time_roll_3",
        "td_success_rate_roll_3": "fighter_1_td_success_rate_roll_3",
    })
)

fighter_2_features = (
    history[
        [
            "fight_id",
            "fighter",

            "td_avg_roll_3",
            "td_acc_roll_3",
            "td_def_roll_3",
            "sub_avg_roll_3",
            "ctrl_time_roll_3",
            "td_success_rate_roll_3",
        ]
    ]
    .rename(columns={
        "fighter": "fighter_2",
        "td_avg_roll_3": "fighter_2_td_avg_roll_3",
        "td_acc_roll_3": "fighter_2_td_acc_roll_3",
        "td_def_roll_3": "fighter_2_td_def_roll_3",
        "sub_avg_roll_3": "fighter_2_sub_avg_roll_3",
        "ctrl_time_roll_3": "fighter_2_ctrl_time_roll_3",
        "td_success_rate_roll_3": "fighter_2_td_success_rate_roll_3",
    })
)

df = df.merge(
    fighter_1_features,
    on=["fight_id", "fighter_1"],
    how="left"
)

df = df.merge(
    fighter_2_features,
    on=["fight_id", "fighter_2"],
    how="left"
)

# Feature Engineering

## Creation

In [ ]:
#Differentials



#Physical
df['height_diff'] = df['fighter_1_height_cm'] - df['fighter_2_height_cm']
df['reach_diff'] = df['fighter_1_reach_cm'] - df['fighter_2_reach_cm']
df['weight_diff'] = df['fighter_1_weight_lbs'] - df['fighter_2_weight_lbs']
df["age_diff"] = (
    (pd.to_datetime(df["event_date"]) - pd.to_datetime(df["fighter_1_dob"])).dt.days -
    (pd.to_datetime(df["event_date"]) - pd.to_datetime(df["fighter_2_dob"])).dt.days
) / 365.25


#Record
df['win_diff'] = (
    df['fighter_1_prior_wins']
    - df['fighter_2_prior_wins']
)

df['loss_diff'] = (
    df['fighter_1_prior_losses']
    - df['fighter_2_prior_losses']
)

df['draw_diff'] = (
    df['fighter_1_prior_draws']
    - df['fighter_2_prior_draws']
)



# Striking (rolling averages already merged)
df["slpm_roll_3_diff"] = (
    df["fighter_1_slpm_roll_3"]
    - df["fighter_2_slpm_roll_3"]
)

df["sapm_roll_3_diff"] = (
    df["fighter_1_sapm_roll_3"]
    - df["fighter_2_sapm_roll_3"]
)

df["str_acc_roll_3_diff"] = (
    df["fighter_1_str_acc_roll_3"]
    - df["fighter_2_str_acc_roll_3"]
)

df["str_def_roll_3_diff"] = (
    df["fighter_1_str_def_roll_3"]
    - df["fighter_2_str_def_roll_3"]
)

# #Grappling
df["td_avg_roll_3_diff"] = (
    df["fighter_1_td_avg_roll_3"]
    - df["fighter_2_td_avg_roll_3"]
)

df["td_acc_roll_3_diff"] = (
    df["fighter_1_td_acc_roll_3"]
    - df["fighter_2_td_acc_roll_3"]
)

df["td_def_roll_3_diff"] = (
    df["fighter_1_td_def_roll_3"]
    - df["fighter_2_td_def_roll_3"]
)

df["sub_avg_roll_3_diff"] = (
    df["fighter_1_sub_avg_roll_3"]
    - df["fighter_2_sub_avg_roll_3"]
)

df["ctrl_time_roll_3_diff"] = (
    df["fighter_1_ctrl_time_roll_3"]
    - df["fighter_2_ctrl_time_roll_3"]
)

df["td_success_rate_roll_3_diff"] = (
    df["fighter_1_td_success_rate_roll_3"]
    - df["fighter_2_td_success_rate_roll_3"]
)

#Perception
df["rank_diff"] = df["fighter_2_rank"] - df["fighter_1_rank"]
df['odds_diff'] = df['fighter_1_odds'] - df['fighter_2_odds']

## Selection

In [ ]:
feature_cols = [
    # Physical
    "height_diff",
    "reach_diff",
    "weight_diff",
    "age_diff",

    # Record
    "win_diff",
    "loss_diff",
    "draw_diff",

    # Striking
    "slpm_roll_3_diff",
    "sapm_roll_3_diff",
    "str_acc_roll_3_diff",
    "str_def_roll_3_diff",

    # Grappling
    "td_avg_roll_3_diff",
    "td_acc_roll_3_diff",
    "td_def_roll_3_diff",
    "sub_avg_roll_3_diff",
    "ctrl_time_roll_3_diff",
    "td_success_rate_roll_3_diff",

    # Perception
    "rank_diff",
    "odds_diff"
]

df = df.sort_values("event_date")


X = df[feature_cols]

y = df["fighter_1_win"]

## Transformation

# Model Construction


### Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix
)
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV, TimeSeriesSplit



In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight='balanced', max_iter=1000))
])


split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train = train_df[feature_cols]
y_train = train_df["fighter_1_win"]

X_test = test_df[feature_cols]
y_test = test_df["fighter_1_win"]

pipeline.fit(X_train, y_train)

preds = pipeline.predict(X_test)
probs = pipeline.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, preds)



In [ ]:
pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight='balanced', max_iter=1000))
])

param_dist = {
    "model__C": [0.001, 0.01, 0.1, 1, 10, 100],
    "model__penalty": ["l2"],
}

tscv = TimeSeriesSplit(n_splits=5)

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_dist,
    n_iter=6,
    scoring="roc_auc",
    cv=tscv,
    random_state=42,
    n_jobs=-1
)

# ---- time-based split ----
split_idx = int(len(df) * 0.8)

train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:]

X_train = train_df[feature_cols]
y_train = train_df["fighter_1_win"]

X_test = test_df[feature_cols]
y_test = test_df["fighter_1_win"]

# ---- hyperparameter search ----
search.fit(X_train, y_train)

# ---- IMPORTANT: use best model, not pipeline ----
best_model = search.best_estimator_

# ---- evaluation ----
preds = best_model.predict(X_test)
probs = best_model.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, preds)
precision = precision_score(y_test, preds)
recall = recall_score(y_test, preds)
f1 = f1_score(y_test, preds)
roc_auc = roc_auc_score(y_test, probs)

print("Best params:", search.best_params_)
print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1:", f1)
print("ROC AUC:", roc_auc)

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))

print("\nClassification Report:")
print(classification_report(y_test, preds))

Best params: {'model__penalty': 'l2', 'model__C': 0.01}
Accuracy: 0.660958904109589
Precision: 0.6993534482758621
Recall: 0.6732365145228216
F1: 0.686046511627907
ROC AUC: 0.7208898519283021

Confusion Matrix:
[[509 279]
 [315 649]]

Classification Report:
              precision    recall  f1-score   support

           0       0.62      0.65      0.63       788
           1       0.70      0.67      0.69       964

    accuracy                           0.66      1752
   macro avg       0.66      0.66      0.66      1752
weighted avg       0.66      0.66      0.66      1752



In [ ]:
y.value_counts()

,count
fighter_1_win,
1,5519
0,3239


In [ ]:
print(f"Accuracy:  {accuracy_score(y_test, preds):.4f}")
print(f"Precision: {precision_score(y_test, preds):.4f}")
print(f"Recall:    {recall_score(y_test, preds):.4f}")
print(f"F1 Score:  {f1_score(y_test, preds):.4f}")
print(classification_report(y_test, preds))
print(f"ROC AUC:   {roc_auc_score(y_test, probs):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, preds))

Accuracy:  0.6610
Precision: 0.6994
Recall:    0.6732
F1 Score:  0.6860
              precision    recall  f1-score   support

           0       0.62      0.65      0.63       788
           1       0.70      0.67      0.69       964

    accuracy                           0.66      1752
   macro avg       0.66      0.66      0.66      1752
weighted avg       0.66      0.66      0.66      1752

ROC AUC:   0.7209

Confusion Matrix:
[[509 279]
 [315 649]]


### Boosted Trees

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Random Forest
rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(random_state=42, class_weight= 'balanced'))
])

rf_pipeline.fit(X_train, y_train)

rf_preds = rf_pipeline.predict(X_test)
rf_probs = rf_pipeline.predict_proba(X_test)[:, 1]


In [ ]:
print(f"Accuracy:  {accuracy_score(y_test, rf_preds):.4f}")
print(f"Precision: {precision_score(y_test, rf_preds):.4f}")
print(f"Recall:    {recall_score(y_test, rf_preds):.4f}")
print(f"F1 Score:  {f1_score(y_test, rf_preds):.4f}")
print(classification_report(y_test, rf_preds))
print(f"ROC AUC:   {roc_auc_score(y_test, rf_probs):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, rf_preds))

Accuracy:  0.6244
Precision: 0.6150
Recall:    0.8485
F1 Score:  0.7132
              precision    recall  f1-score   support

           0       0.65      0.35      0.46       788
           1       0.62      0.85      0.71       964

    accuracy                           0.62      1752
   macro avg       0.63      0.60      0.58      1752
weighted avg       0.63      0.62      0.60      1752

ROC AUC:   0.6760

Confusion Matrix:
[[276 512]
 [146 818]]


In [ ]:
from xgboost import XGBClassifier

xgb_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", XGBClassifier(
        n_estimators=500,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    min_child_weight=5,
    gamma=0.1,
    random_state=42,
    eval_metric="logloss",
    class_weight='balanced'

    ))
])

xgb_pipeline.fit(X_train, y_train)

xgb_preds = xgb_pipeline.predict(X_test)
xgb_probs = xgb_pipeline.predict_proba(X_test)[:, 1]



/usr/local/lib/python3.12/dist-packages/xgboost/training.py:200: UserWarning: [06:17:52] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "class_weight" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [ ]:
print(f"Accuracy:  {accuracy_score(y_test, xgb_preds):.4f}")
print(f"Precision: {precision_score(y_test, xgb_preds):.4f}")
print(f"Recall:    {recall_score(y_test, xgb_preds):.4f}")
print(f"F1 Score:  {f1_score(y_test, xgb_preds):.4f}")
print(classification_report(y_test, xgb_preds))
print(f"ROC AUC:   {roc_auc_score(y_test, xgb_probs):.4f}")

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, xgb_preds))

Accuracy:  0.6330
Precision: 0.6281
Recall:    0.8164
F1 Score:  0.7100
              precision    recall  f1-score   support

           0       0.65      0.41      0.50       788
           1       0.63      0.82      0.71       964

    accuracy                           0.63      1752
   macro avg       0.64      0.61      0.61      1752
weighted avg       0.64      0.63      0.62      1752

ROC AUC:   0.6754

Confusion Matrix:
[[322 466]
 [177 787]]


d

In [ ]:
list(df.columns)

['fight_id',
 'fight_url',
 'event_url',
 'event_name',
 'event_date',
 'location_city',
 'location_state',
 'location_country',
 'referee',
 'weight_class',
 'gender',
 'title_fight',
 'num_rounds',
 'fighter_1',
 'fighter_2',
 'fighter_1_url',
 'fighter_2_url',
 'winner',
 'result',
 'result_details',
 'finish_round',
 'finish_time',
 'fighter_1_height_cm',
 'fighter_1_weight_lbs',
 'fighter_1_reach_cm',
 'fighter_1_stance',
 'fighter_1_dob',
 'fighter_2_height_cm',
 'fighter_2_weight_lbs',
 'fighter_2_reach_cm',
 'fighter_2_stance',
 'fighter_2_dob',
 'fighter_1_wins',
 'fighter_1_losses',
 'fighter_1_draws',
 'fighter_1_slpm',
 'fighter_1_str_acc',
 'fighter_1_sapm',
 'fighter_1_str_def',
 'fighter_1_td_avg',
 'fighter_1_td_acc',
 'fighter_1_td_def',
 'fighter_1_sub_avg',
 'fighter_2_wins',
 'fighter_2_losses',
 'fighter_2_draws',
 'fighter_2_slpm',
 'fighter_2_str_acc',
 'fighter_2_sapm',
 'fighter_2_str_def',
 'fighter_2_td_avg',
 'fighter_2_td_acc',
 'fighter_2_td_def',
 'fighte